# Sistema de búsqueda de estudiantes

Tienes un archivo con 10,000 estudiantes (nombre, edad,
promedio). Necesitas implementar un sistema que
permita:
1. Buscar un estudiante por su ID (número de matrícula)
2. Insertar nuevos estudiantes
3. Listar todos los estudiantes en orden por ID

Se desean organizar los datos por medio de: listas (nativas de python), árboles binarios tradicionales y árboles B+ para comparar sus desempeños sobre operaciones CRUD.

In [59]:
# Librerías relevantes
from faker import Faker
import random
import time
from statistics import mean
from abc import ABC, abstractmethod

## Funciones y código relevante que será reutilizado durante todo el reto

In [60]:
class Estructura(ABC):
    """
        Clase abstracta, sirve para mantener la coherencia de las clases concretas de árbol binario, gestor de lista y árbol B+
        Todas deben mantener el contrato definido por los métodos de esta clase.
    """
    @abstractmethod
    def insertar(self, estudiante:dict) -> dict:
        pass

    @abstractmethod
    def buscar(self, id_estudiante:int) -> str:
        pass

    @abstractmethod
    def listar(self) -> str:
        pass

In [61]:
# Función generadora de estudiantes, en órden
def generar_estudiantes_orden(cantidad : int) -> list:
    faker = Faker() # Instanciamos faker para generar los datos
    estudiantes = []

    for i in range(1, cantidad + 1):
        estudiante = {"id": i, "nombre" : faker.name(), "promedio" : round(random.uniform(0, 10), 1)}
        estudiantes.append(estudiante)

    return estudiantes


# Función que genera estudiantes pero la lista no respeta el órden por id
def generar_estudiantes_sin_orden(cantidad : int) -> list:
    estudiantes = generar_estudiantes_orden(cantidad)
    random.shuffle(estudiantes)
    return estudiantes

# Función encargadd de medir el tiempo de búsqueda sobre una muestra de id's
def medir_tiempo_busqueda(estructura : Estructura, muestra):
    inicio = time.perf_counter()
    
    for id_objetivo in muestra:
        estructura.buscar(id_objetivo)

    fin = time.perf_counter()

    return fin - inicio

## Sistema de búsqueda, inserción y listado de todos los estudiantes

In [62]:
class GestorListaEstudiantes(Estructura):
    """
        Esta clase se encarga de gestionar las operaciones de inserción, búsqueda y listado de los estudiantes
        cuando la estructura de datos en cuestión es una lista nativa de Python.
    """
    def __init__(self, datos : list):
        self.datos = datos

    def insertar(self, estudiante:dict) -> dict:
        self.datos.append(estudiante)
        return estudiante

    def buscar(self, id_estudiante:int) -> str:
        for estudiante in self.datos:
            if estudiante["id"] == id_estudiante:
                resultado = ""
                resultado += "Encontrado :\n"
                for clave, valor in estudiante.items():
                    resultado += f"{clave} : {valor} | "
                return resultado

        return f"Estudiante con id: {id_estudiante} no fue encontrado."

    def listar(self) -> str:
        listado = ""
        for estudiante in self.datos:
            listado += "Estudiante:\n"
            listado += "["
            for clave, valor in estudiante.items():
                listado += f"{clave} : {valor} |"
            listado += "]\n"

        return listado      

In [63]:
class ABB(Estructura):
    """
        Esta clase representa una implementación básica de un árbol binario de búsqueda (ABB),
        hereda de la clase abstracta Estrucrura, para mantener los métodos de forma consistente con las
        otras estructuras de datos.

        Las operaciones de inserción, búsqueda y mostrar los datos son implementaciones basadas en los algoritmos
        del libro Introduction to Algorithms de Cormen.
    """

    class Nodo:
        """
            Esta clase representa un nodo de árbol binario de búsqueda.
            Para este reto, se decidió implementar el árbol a modo de lista ligada,
            es decir, el nodo guarda referencias hacia sus hijos.
        """
        def __init__(self, estudiante :dict, hijo_izquierdo : "ABB.Nodo" = None, hijo_derecho : "ABB.Nodo" = None):
            self.hijo_izquierdo = hijo_izquierdo
            self.hijo_derecho = hijo_derecho
            self.estudiante = estudiante

    def __init__(self):
        self.raiz = None #Raíz inicia siendo nula

    def insertar(self, estudiante :dict) -> dict:
        """
            Dado un diccionario que representa un estudiante, se inserta el nodo correspondiente en el árbol.
            Este es el algoritmo estándar (versión iterativa) de inserción en un árbol binario, adaptado del libro Introduction
            To Algorithms de Cormen.

            El procedimiento es el que sigue:
                1. nodo_actual se utiliza para recorrer el árbol y nodo_previo se utiliza para almacenar el nodo inmediatamente anterior
                    al nodo actual.
                2. Siempre que nodo_actual no sea nulo, desde él se decide si la inserción continúa por la izquierda (valor <=) o la derecha.
                3. Se actualiza el nodo_actual a su hijo izquierdo o derecho y se guarda en nodo_previo la referencia a él.
                4. Una vez que se llegue a una hoja, nodo_actual será nulo y nodo_previo será el padre del valor a insertar.
                5. Finalmente, se decide si el valor se inserta a la izquierda o a la derecha del nodo_previo.

        """
        nodo_actual = self.raiz
        nodo_previo = None
        while nodo_actual is not None:
            nodo_previo = nodo_actual
            if estudiante["id"] <= nodo_actual.estudiante["id"]:
                nodo_actual = nodo_actual.hijo_izquierdo
            else:
                nodo_actual = nodo_actual.hijo_derecho

        if nodo_previo is None:
            self.raiz = self.Nodo(estudiante=estudiante) # Caso especial: árbol totalmente vacío.
        elif estudiante["id"] <= nodo_previo.estudiante["id"]:
            nodo_previo.hijo_izquierdo = self.Nodo(estudiante=estudiante)
        else:
            nodo_previo.hijo_derecho = self.Nodo(estudiante=estudiante)
        
        return estudiante

    def buscar(self, id_estudiante : int) -> str:
        """
            Dado un id de estudiante, recorre el árbol para encontrar el nodo correspondiente.
            Este es el algoritmo estándar (versión iterativa) de búsqueda en un árbol binario, adaptado del libro
            Introduction To Algorithms de Cormen.

            El procedimiento es el que sigue:
                1. nodo_actual se utiliza para recorrer el árbol, comenzando desde la raíz.
                2. En cada iteración, si el id buscado coincide con el del nodo_actual, el ciclo termina (encontrado).
                3. Si no coincide, se decide si continuar por la izquierda (id buscado <=) o por la derecha,
                    aprovechando la propiedad de orden del ABB para descartar la mitad del subárbol en cada paso.
                4. El ciclo también termina si nodo_actual llega a ser nulo, lo que significa que el id
                    no existe en el árbol.
                5. Finalmente, se construye el string de resultado según si el nodo fue encontrado o no.

        """
        nodo_actual = self.raiz
        while nodo_actual is not None and id_estudiante != nodo_actual.estudiante["id"]:
            if id_estudiante <= nodo_actual.estudiante["id"]:
                nodo_actual = nodo_actual.hijo_izquierdo
            else:
                nodo_actual = nodo_actual.hijo_derecho

        resultado = ""
        if nodo_actual is not None:
            resultado += "Encontrado :\n"
            for clave, valor in nodo_actual.estudiante.items():
                resultado += f"{clave} : {valor} | "
        else:
            resultado = f"Estudiante con id: {id_estudiante} no fue encontrado."
    
        return resultado

    def listar(self) -> str:
        """
            Retorna un string con todos los estudiantes del árbol, ordenados de forma ascendente por id.

            Este método es un simple punto de entrada que delega el trabajo real en _inorden, el cual
            realiza un recorrido in-order (izquierda, nodo, derecha) sobre el árbol. Gracias a la propiedad
            de orden del ABB, este tipo de recorrido garantiza que los estudiantes se visiten en orden
            ascendente por id, sin necesidad de un paso adicional de ordenamiento (a diferencia de la
            lista, donde listar en orden requiere un sort completo).

            Se usa una lista auxiliar mutable (resultado) para ir acumulando los fragmentos de texto y luego
            unirlos con "".join(), en vez de concatenar strings directamente en cada llamada recursiva,
            ya que la concatenación repetida de strings es más costosa en tiempo (cada concatenación crea
            un nuevo string en memoria).
        """
        resultado = []
        self._inorden(self.raiz, resultado)
        return "".join(resultado)

    def _inorden(self, nodo : Nodo, resultado : list):
        """
            Recorrido in-order recursivo sobre el árbol, usado como método auxiliar de listar().

            El recorrido in-order visita primero el subárbol izquierdo, luego el nodo actual, y finalmente
            el subárbol derecho. Por la propiedad de orden de un ABB (todo lo que está a la izquierda de
            un nodo es menor, y todo lo que está a la derecha es mayor), este orden de visita produce
            los ids en secuencia ascendente.

            Caso base: si nodo es None, no hay nada que recorrer ni agregar, y la recursión termina
            en esa rama.
        """
        if nodo is None:
            return

        self._inorden(nodo.hijo_izquierdo, resultado)

        resultado.append("Estudiante:\n")
        datos = "["
        for clave, valor in nodo.estudiante.items():
            datos += f"{clave} : {valor} | "
        datos += "]\n"

        resultado.append(datos)

        self._inorden(nodo.hijo_derecho, resultado)

    def construir_arbol(self, datos : list) -> None:
        """
            Construye el árbol completo a partir de una lista de estudiantes, insertándolos uno por uno
            en el orden en que aparecen en la lista.
        """
        for estudiante in datos:
            self.insertar(estudiante)

### Pruebas y mediciones

In [64]:
# Prueba 1: datos ordenados por id en una lista

# Generamos la lista de estudiante, ordenada por id de forma creciente
lista_ordenada = generar_estudiantes_orden(cantidad= 10000)

# Instanciamos el gestor de la lista
gestor = GestorListaEstudiantes(datos=lista_ordenada)

# Realizamos 200 mediciones para hallar el promedio de tiempo que tarda el sistema en buscar 100 id's 
# escogidos aleatoriamente, cada intento se hace sobre una muestra de id's diferentes.
resultados = []
for _ in range(200):
    tiempo = medir_tiempo_busqueda(gestor, random.sample(range(1, len(lista_ordenada) + 1), 100))
    resultados.append(tiempo)

print(f"El promedio de tiempo (s) de búsqueda de 100 estudiantes por id (lista ordenada por id) es: {mean(resultados)}")


# Prueba 2: datos desordenados en una lista
lista_desordenada = generar_estudiantes_sin_orden(cantidad=10000)

# Lo siguiente es igual al código previo:
gestor = GestorListaEstudiantes(datos=lista_desordenada)
resultados = []
for _ in range(200):
    tiempo = medir_tiempo_busqueda(gestor, random.sample(range(1, len(lista_desordenada) + 1), 100))
    resultados.append(tiempo)

print(f"El promedio de tiempo (s) de búsqueda de 100 estudiantes por id (lista desordenada) es: {mean(resultados)}")

El promedio de tiempo (s) de búsqueda de 100 estudiantes por id (lista ordenada por id) es: 0.13975363650009057
El promedio de tiempo (s) de búsqueda de 100 estudiantes por id (lista desordenada) es: 0.15539423700002772


In [ ]:
# Prueba 3: datos ordenados por id en un árbol binario

# Generamos la lista de estudiante, ordenada por id de forma creciente
lista_ordenada = generar_estudiantes_orden(cantidad= 10000)

# Instanciamos el árbol binario
arbol_abb = ABB()
arbol_abb.construir_arbol(datos=lista_ordenada)

# Realizamos 200 mediciones para hallar el promedio de tiempo que tarda el sistema en buscar 100 id's 
# escogidos aleatoriamente, cada intento se hace sobre una muestra de id's diferentes.
resultados = []
for _ in range(200):
    tiempo = medir_tiempo_busqueda(arbol_abb, random.sample(range(1, len(lista_ordenada) + 1), 100))
    resultados.append(tiempo)

print(f"El promedio de tiempo (s) de búsqueda de 100 estudiantes por id (árbol binario con lista ordenada por id) es: {mean(resultados)}")


# Prueba 4: datos desordenados en una lista dentro de un árbol binario
lista_desordenada = generar_estudiantes_sin_orden(cantidad=10000)

# Lo siguiente es igual al código previo:
arbol_abb = ABB()
arbol_abb.construir_arbol(datos=lista_desordenada)
resultados = []
for _ in range(200):
    tiempo = medir_tiempo_busqueda(arbol_abb, random.sample(range(1, len(lista_desordenada) + 1), 100))
    resultados.append(tiempo)

print(f"El promedio de tiempo (s) de búsqueda de 100 estudiantes por id (árbol binario con lista desordenada) es: {mean(resultados)}")

El promedio de tiempo (s) de búsqueda de 100 estudiantes por id (árbol binario con lista ordenada por id) es: 0.21786138249986833
El promedio de tiempo (s) de búsqueda de 100 estudiantes por id (árbol binario con lista desordenada) es: 0.0013659684998674493
